# Block 2 · Regression baselines: linear, logistic, and the honest split
> Data Mining · SGH Warsaw School of Economics (SMMD-ADA)
> Course anchor dataset: retail-credit / PD portfolio

**Business framing.** Same lender, same portfolio as Block 1 — but today we predict. A linear warm-up (savings from income), then the model that runs the world's credit departments: a **logistic PD baseline**, built leakage-safe in a pipeline, read through coefficients and odds ratios, and judged the honest way — on data it has never seen.

**Learning goals for this lab**
1. Split data with `train_test_split` (stratified, seeded) and understand why.
2. Fit and evaluate a linear regression; read residual plots.
3. Build a logistic PD baseline inside a `Pipeline`.
4. Interpret coefficients as odds ratios.
5. Play the threshold game: confusion matrices, precision, recall — and why 0.5 is nobody's law.

**How to work.** Run cells top to bottom. Before you trust any result: **Kernel → Restart & Run All.**

In [ ]:
import sys
sys.path.append("../lecture_1")   # the shared course data loader lives there

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option('display.max_columns', 30)

from finance_data import load_taiwan, taiwan_features
df = load_taiwan()
df.shape

## 1. Features, Block-1 style
We assemble a feature matrix using only what Block 1 taught us is honest: raw application-time numerics, median imputation **with indicators**, and the engineered flags. Note who is *not* invited: `collections_contact` (the time traveller) and `customer_id` (an identifier, never a feature).

In [ ]:
X = taiwan_features(df)   # the shared course feature matrix (Blocks 1-2 craft)
y = df["default"]
X.head()

## 2. The split
70/30, **stratified** (same default rate in both halves), **seeded** (reproducible — Block 1's habit). From this cell on, the test set is locked in a drawer: it answers one question, once, at the end.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)

print(f"train: {len(X_train)} rows, default rate {y_train.mean():.1%}")
print(f"test:  {len(X_test)} rows, default rate {y_test.mean():.1%}")

## 3. Linear warm-up: payments from bills
How much of last month's bill shows up in this month's payment? Same regression you know from econometrics, judged the prediction way. Log–log (Block 1's lens for money), so the slope is an **elasticity**.

In [ ]:
from sklearn.linear_model import LinearRegression

# September payment vs August bill, among paying clients, log10 both sides
d = df[(df["pay_amt_sep"] > 0) & (df["bill_amt_aug"] > 0)]
d_tr = d.loc[d.index.intersection(X_train.index)]
d_te = d.loc[d.index.intersection(X_test.index)]

lin = LinearRegression()
lin.fit(np.log10(d_tr[["bill_amt_aug"]]), np.log10(d_tr["pay_amt_sep"]))
print(f"slope (elasticity): {lin.coef_[0]:.2f}   # 1% bigger bill ~ {lin.coef_[0]:.2f}% more payment")
print(f"intercept:          {lin.intercept_:.2f}")

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

pred_te = lin.predict(np.log10(d_te[["bill_amt_aug"]]))
true_te = np.log10(d_te["pay_amt_sep"])
print(f"R^2 on TEST:  {r2_score(true_te, pred_te):.3f}")
print(f"MAE on TEST:  {mean_absolute_error(true_te, pred_te):.3f} (log10 units)")

$R^2 \approx 0.33$ on test. The economics is in the slope: only ~0.39% of payment per 1% of bill — revolvers finance bill growth rather than repaying it. (The reverse — a suspiciously *high* $R^2$ — is what should wake you up at night: leakage or a target twin.)

In [ ]:
# Residuals vs fitted — the plot that tells the truth about a regression.
fitted = lin.predict(np.log10(d_tr[["bill_amt_aug"]]))
resid = np.log10(d_tr["pay_amt_sep"]) - fitted

plt.figure(figsize=(7, 3.2))
plt.scatter(fitted, resid, s=6, alpha=0.3)
plt.axhline(0, color="gray")
plt.xlabel("fitted log10 payment"); plt.ylabel("residual")
plt.title("Want: a structureless band around zero")
plt.tight_layout(); plt.show()

## 4. The PD baseline: logistic regression in a pipeline
The pipeline welds the scaler to the model, so the scaler's mean and SD are fitted on **training data only** — the leakage-safe pattern that becomes non-negotiable next week inside cross-validation.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
pipe.fit(X_train, y_train)

proba_test = pipe.predict_proba(X_test)[:, 1]   # P(default) per customer
proba_test[:5].round(3)

## 5. Reading the model: coefficients and odds ratios
Standardised coefficients (the scaler did that for us): each is the change in **log-odds per one standard deviation** of the feature. Exponentiate to speak the language of credit: odds multipliers.

In [ ]:
coefs = pd.Series(pipe.named_steps["logisticregression"].coef_[0], index=X.columns)
summary = pd.DataFrame({
    "coef (log-odds / SD)": coefs.round(3),
    "odds ratio e^coef":    np.exp(coefs).round(3),
}).sort_values("coef (log-odds / SD)")
summary

In [ ]:
colors = ["#C83C28" if c < 0 else "#2E6DA4" for c in summary["coef (log-odds / SD)"]]
plt.figure(figsize=(7, 3.6))
plt.barh(summary.index, summary["coef (log-odds / SD)"], color=colors)
plt.axvline(0, color="#5B6570")
plt.xlabel("standardised coefficient (log-odds per 1 SD)")
plt.tight_layout(); plt.show()

Read the top and bottom rows aloud:
* **Payment history is king**: the engineered delay counter `months_delayed_6m` (+0.58, odds ×1.8 per SD) and the recent status (+0.37) tower over everything — Block 1's craft on real data.
* `limit_bal` at −0.23: higher limit, *lower* risk — the bank's old screening leaking into a coefficient (Block 1's selection bias).

*Sanity check the signs against your Block 1 EDA — a coefficient that contradicts the bivariate plot deserves an investigation, not a shrug.*

### 5a. One customer, step by step
Logistic regression is linear in log-odds, so every prediction decomposes **exactly**: each feature contributes `coef × standardised value`, the contributions plus the intercept sum to $z$, and $\sigma(z)$ is the PD. This "receipt" is where adverse-action reasons come from — and it is the exact, free version of what SHAP (Block 3) approximates for less polite models.

In [ ]:
scaler = pipe.named_steps["standardscaler"]
lr = pipe.named_steps["logisticregression"]

i = int(np.argmin(np.abs(proba_test - 0.65)))   # a deterministic, interesting customer
xs = scaler.transform(X_test.iloc[[i]])[0]
contrib = pd.Series(lr.coef_[0] * xs, index=X.columns).sort_values(ascending=False)

z = lr.intercept_[0] + contrib.sum()
p = 1 / (1 + np.exp(-z))
print(contrib.round(3).to_string())
print(f"\nintercept {lr.intercept_[0]:+.3f}  ->  z = {z:+.3f}  ->  PD = {p:.1%}")
print("Largest positive contributions = this customer's adverse-action reasons.")

One subtlety before you present odds ratios to anyone: they are constant **by construction**, but probability effects are not. The same ×1.54 odds multiplier moves a 10% PD by +4.6 pp, a 50% PD by +10.6 pp, and a 90% PD by +3.3 pp — the sigmoid is steep in the middle, flat in the tails (marginal effect $\beta_j\,p(1-p)$). Say which scale you are on, and report probability changes in **percentage points**.

## 6. The threshold game
`predict` silently applies a 0.5 threshold. In credit, the threshold is a **business decision** — so we play it out. First, the accuracy trap on our own numbers:

In [ ]:
from sklearn.metrics import accuracy_score

acc_model = accuracy_score(y_test, proba_test >= 0.5)
acc_lazy = accuracy_score(y_test, np.zeros_like(y_test))   # always 'repaid'
print(f"accuracy, our model @0.5:      {acc_model:.2%}")
print(f"accuracy, always-'repaid':     {acc_lazy:.2%}")
print("Three points better than doing nothing. Accuracy is the wrong lens — open the box:")

In [ ]:
from sklearn.metrics import confusion_matrix

def show_confusion(threshold: float) -> None:
    pred = (proba_test >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    prec = tp / (tp + fp) if tp + fp else float("nan")
    rec = tp / (tp + fn)
    print(f"threshold {threshold:.2f}:  TP={tp:4d}  FP={fp:4d}  FN={fn:4d}  TN={tn:4d}"
          f"   precision={prec:.2f}  recall={rec:.2f}")

show_confusion(0.50)
show_confusion(0.30)

Lowering the threshold from 0.50 to 0.30 catches far more defaulters (recall jumps) at the price of declining more good customers (precision falls). Neither is "correct" — the right cut-off depends on what a missed defaulter costs versus a lost customer, and that ratio belongs to the business.

In [ ]:
# The whole trade-off in one picture.
thresholds = np.linspace(0.05, 0.75, 50)
prec, rec = [], []
for t in thresholds:
    pred = proba_test >= t
    tp = ((pred == 1) & (y_test == 1)).sum()
    fp = ((pred == 1) & (y_test == 0)).sum()
    fn = ((pred == 0) & (y_test == 1)).sum()
    prec.append(tp / (tp + fp) if tp + fp else np.nan)
    rec.append(tp / (tp + fn))

plt.figure(figsize=(7, 3.2))
plt.plot(thresholds, prec, label="precision", color="#2E6DA4")
plt.plot(thresholds, rec, label="recall", color="#C83C28")
plt.axvline(0.5, color="gray", linestyle=":")
plt.xlabel("decision threshold"); plt.ylabel("metric")
plt.legend(); plt.tight_layout(); plt.show()

### 6a. One number that ignores the threshold: AUC
Every metric so far depended on where we put the cut-off. **AUC** (area under the ROC curve) sidesteps that: it is the probability that the model ranks a randomly chosen defaulter above a randomly chosen good customer. 0.5 = coin flip, 1.0 = perfect ranking. It scores the *ordering*, not any particular threshold — which is why we'll use it to compare models in Block 3 (where the ROC curve gets its full story). You need it for exercise 4:

In [ ]:
from sklearn.metrics import roc_auc_score

print(f"test AUC of the baseline: {roc_auc_score(y_test, proba_test):.3f}")

## 7. Optional extras: touch the leash
`LogisticRegression` regularises **by default** (`C=1`, and `C = 1/α` — bigger C means weaker penalty). Watch the coefficients react to the leash:

In [ ]:
rows = {}
for C in [0.01, 0.1, 1, 10]:
    p = make_pipeline(StandardScaler(),
                      LogisticRegression(C=C, max_iter=2000)).fit(X_train, y_train)
    rows[f"C={C}"] = p.named_steps["logisticregression"].coef_[0]
pd.DataFrame(rows, index=X.columns).round(3)

Small `C` (strong penalty) shrinks everything toward zero; the *ordering* of the important features barely moves. That stability is one more reason logistic regression makes such a trustworthy baseline.

## 8. Exercises
1. Add `education` via one-hot encoding (`pd.get_dummies`, remember to drop one level). Does test accuracy move? Do any new coefficients earn their place?
2. Compute the confusion matrix at the threshold that gives **recall ≥ 0.5**. What did precision cost?
3. Fit the linear warm-up with `log10(limit_bal)` added. Does test $R^2$ improve? Interpret both slopes.
4. Refit the logistic model **without** `months_delayed_6m`. How much does test AUC change — and which features absorb its role? (Recall: correlated teammates.)
5. Rebuild `utilisation` with the *April* bill instead of September. Does the model care which month you use? What does that say about behavioural persistence?

*Hand-in for the project team: your baseline's confusion matrix at two thresholds, plus three sentences: which features drive it, which number you trust most, and why.*

**"Done" for today** = you can defend your baseline: features used, one number you trust, and *why* that number.

In [ ]:
# scratch cell for the exercises